###Data Modeling Decisions

####Fact Sales

**Table:** fact_sales

**Grain:** One row represents one product item within an order.

**Source:** clean_order_items joined with clean_orders.

**Measures:**
- price
- freight_value
- total_item_value

####Fact Order Payment

**Table:** fact_order_payment

**Grain:** One row represents one order with aggregated payment information.

**Source:** clean_order_payments.

**Measures:**
- total_payment_value
- payment_transaction_count
- max_payment_installments

####Dimensions

####dim_customer
One row per customer.

####dim_product
One row per product.

####dim_order
One row per order.

####Important Modeling Decision

Payment records were aggregated at the order level before being connected to the analytical model.

This prevents payment values from being duplicated when an order contains multiple order items.

####clean_orders → clean_customers

In [0]:
%sql

SELECT
    COUNT(*) AS unmatched_orders
FROM clean_orders o
LEFT JOIN clean_customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

#####relationship: Order Items → Orders

In [0]:
%sql

SELECT
    COUNT(*) AS unmatched_order_items
FROM clean_order_items oi
LEFT JOIN clean_orders o
    ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;

####Order Items → Products

In [0]:
%sql

SELECT
    COUNT(*) AS unmatched_products
FROM clean_order_items oi
LEFT JOIN clean_products p
    ON oi.product_id = p.product_id
WHERE p.product_id IS NULL;

####relationship : Payments → Orders

In [0]:
%sql

SELECT
    COUNT(*) AS unmatched_payments
FROM clean_order_payments op
LEFT JOIN clean_orders o
    ON op.order_id = o.order_id
WHERE o.order_id IS NULL;

####creating dim_customer

In [0]:
%sql

CREATE OR REPLACE TABLE dim_customer AS
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM clean_customers;

In [0]:
%sql

SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM dim_customer;

####Creating dim_product

In [0]:
%sql

CREATE OR REPLACE TABLE dim_product AS
SELECT
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM clean_products;

In [0]:
%sql

SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT product_id) AS unique_products
FROM dim_product;

####Creating dim_order

In [0]:
%sql

CREATE OR REPLACE TABLE dim_order AS
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM clean_orders;

In [0]:
%sql

SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders
FROM dim_order;

In [0]:
%sql

SELECT
    COUNT(*) AS total_order_items,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT product_id) AS unique_products
FROM clean_order_items;

####Building the initial fact_sales

In [0]:
%sql

CREATE OR REPLACE TABLE fact_sales AS
SELECT
    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    oi.price,
    oi.freight_value,
    oi.price + oi.freight_value AS total_item_value
FROM clean_order_items oi
INNER JOIN clean_orders o
    ON oi.order_id = o.order_id;

In [0]:
%sql

SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT product_id) AS unique_products
FROM fact_sales;

####Validating the sales measures

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(price) AS total_product_sales,
    SUM(freight_value) AS total_freight,
    SUM(total_item_value) AS total_sales_value,
    MIN(price) AS min_price,
    MAX(price) AS max_price
FROM fact_sales;

####Creating order_payment_summary

In [0]:
%sql

CREATE OR REPLACE TABLE order_payment_summary AS
SELECT
    order_id,
    SUM(payment_value) AS total_payment_value,
    COUNT(*) AS payment_transaction_count,
    MAX(payment_installments) AS max_payment_installments
FROM clean_order_payments
GROUP BY order_id;

In [0]:
%sql

SELECT
    COUNT(*) AS payment_orders,
    COUNT(DISTINCT order_id) AS unique_payment_orders,
    SUM(total_payment_value) AS total_payment_value
FROM order_payment_summary;

####verifying the one missing payment order

In [0]:
%sql

SELECT
    COUNT(*) AS orders_without_payment
FROM clean_orders o
LEFT JOIN order_payment_summary p
    ON o.order_id = p.order_id
WHERE p.order_id IS NULL;

####fact_order_payment

In [0]:
%sql

CREATE OR REPLACE TABLE fact_order_payment AS
SELECT
    order_id,
    total_payment_value,
    payment_transaction_count,
    max_payment_installments
FROM order_payment_summary;

In [0]:
%sql

SELECT
    COUNT(*) AS total_payment_rows,
    COUNT(DISTINCT order_id) AS unique_payment_orders
FROM fact_order_payment;